#### Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Add the parent directory (or another appropriate path) to sys.path so Python can find Exclusion_functions
notebook_dir = os.path.dirname(os.path.abspath('04_Optimization.ipynb'))
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..', '..', '..'))
function_dir = os.path.abspath(os.path.join(parent_dir, 'Function_Files'))
if function_dir not in sys.path:
    sys.path.append(function_dir)

import pandas as pd
from openai_api import OpenAIAgent
import nest_asyncio
nest_asyncio.apply()
import Load_Isolate_functions as lif
import Classification_functions as cf
import datetime
import pyodbc
from datetime import datetime, timedelta

#### Variables

In [3]:
CATEGORY = "Cups"
today = datetime.now().strftime('%m.%d.%Y')
ip_path = f"{parent_dir}\\Data\\"
ip_path_cat = f"{parent_dir}\\Data\\{CATEGORY}\\"
OP_PATH = f"{parent_dir}\\Data\\{CATEGORY}\\Output\\"

In [4]:
today

'02.05.2026'

#### Read in data
sfy = salsify, to_keep = files received from ID

In [8]:
#Read in files, define variables.
sfy = pd.read_excel(f"{ip_path}All Salsify Items.xlsx", sheet_name='in', skiprows=1)

# Extract this from Fornax in the same format

In [9]:
sfy.columns

Index(['Product ID', 'Product Name', 'S2K Item Number', 'Description 1',
       'Description 2', 'Selling Unit of Measure (Long)',
       'Gap-Fill Strategy Group', 'Associated Product ID', 'Keywords',
       'Delete Code',
       ...
       'Composting Manufacturing Alliance (CMA) Certification',
       'Green Restaurant Association Certification',
       'Green Seal Product Certification',
       'Center for Resource Solutions Green-e Certification', 'Greensafe',
       'Manufacturer Sustainability Certifications', 'Complimentary Items',
       'Substitution Items', 'Similar Items', 'Marketing Bullets'],
      dtype='object', length=408)

In [1]:
import pyodbc
import pandas as pd

conn_str = (
    r'DRIVER=SQL Server;'
    r'SERVER=ibp-db01;'
    r'DATABASE=fornax;'
)

conn = pyodbc.connect(conn_str)

sql_query = """
    SELECT
        TABLE_SCHEMA,
        TABLE_NAME,
        TABLE_TYPE
    FROM INFORMATION_SCHEMA.TABLES
    ORDER BY TABLE_SCHEMA, TABLE_NAME
"""

tables_df = pd.read_sql(sql_query, conn)
conn.close()

print(f"Found {len(tables_df)} tables/views")
tables_df

ModuleNotFoundError: No module named 'pyodbc'

In [2]:
# Load Salsify data from database
import pyodbc
import pandas as pd

# Define connection string
conn_str = (
    r'DRIVER=SQL Server;'
    r'SERVER=ibp-db01;'
    r'DATABASE=fornax;'
)

# Create connection
conn = pyodbc.connect(conn_str)

# Execute SQL query and fetch result into pandas dataframe
# sql_query = ("""
#              select
#                 salsify_id,
#                 ProductID,
#                 ManufacturerProductNumber,
#                 ProductFamily,
#                 ProductName,
#                 ProductType,
#                 Manufacturer,
#                 Brand,
#                 BrandedProductName
#              from salsify.products""")

sql_query_2 = ("""
             select
                TOP(10) *
             from salsify.products""")


salsify_df = pd.read_sql(sql_query_2, conn)

# Close connection
conn.close()

print(f"Loaded {len(salsify_df)} records from Salsify")
salsify_df.head()

Loaded 10 records from Salsify


C:\Users\MGadupudi\AppData\Local\Temp\ipykernel_41008\1253187894.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  salsify_df = pd.read_sql(sql_query_2, conn)


,salsify_id,salsify_created_at,salsify_updated_at,salsify_version,salsify_profile_asset_id,salsify_system_id,Color,ESPChemical,Manufacturer,Material,...,APEFree,ProcessedChlorineFree,Recyclable,RapidlyRenewableResource,Compostable,GreenProductGuide,_Hash,_FileName,_LineNumber,_RunID
0,10,2021-10-14 11:52:33.503,2026-04-30 12:16:16.777,243,,s-1554d71e-c272-46a6-8878-764b0f28ee1d,,,,,...,,,Check Locally,,,,b'\x92\xfc\xa4I\xf2G\xb0\xc8',products.txt,40623,-1
1,1000,2022-04-05 09:25:00.017,2026-04-30 12:19:56.070,96,,s-4e78bcd7-0f22-4c6d-9ded-cc2d7e496fec,,,,Plastic,...,,,No,No,No,,b'R\x80S\x15\x9d/m\x08',products.txt,138861,-1
2,10000,2021-10-14 11:52:34.227,2026-04-30 12:24:28.017,1061,,s-a4fa8ed1-e74e-49b2-96a4-1e6d5eeca166,"White,Green",,Georgia Pacific,Paper,...,false,false,Check Locally,No,No,Green Certified,b'N\xb7\\\xc7\xa6\x14V\xac',products.txt,283889,-1
3,100000,2021-10-14 11:53:18.377,2026-05-19 11:10:27.763,280,,s-afe38d05-94cf-46c2-b53d-9a990086c06f,Gold,,3M,Synthetic Fiber,...,,,No,No,No,,b'\xaf\xf2\xf2f;9\x90:',products.txt,302241,-1
4,100001,2021-10-14 11:53:18.377,2026-05-01 04:31:56.117,247,,s-ae064549-0cd8-4542-ace2-34b6cc379e9c,Red,,3M,Synthetic Fiber,...,false,false,No,No,No,,"b'\xbc""\x05\xc8\x1a,c\x8a'",products.txt,299113,-1


In [3]:
list(salsify_df.columns)

['salsify_id',
 'salsify_created_at',
 'salsify_updated_at',
 'salsify_version',
 'salsify_profile_asset_id',
 'salsify_system_id',
 'Color',
 'ESPChemical',
 'Manufacturer',
 'Material',
 'ProductFeatures',
 'ProductID',
 'ProductName',
 'ProductType',
 'ProductUsage',
 'SustainableProductFeatures',
 'SustainableProducts',
 'UNSPSC',
 'WebsiteCategoryL1Reference',
 'WebsiteCategoryL2Reference',
 'WebsiteCategoryL3Reference',
 'BagLength_IN_',
 'BagWidth_IN_',
 'BiodegradableProductsInstitute_BPI_Certification',
 'Brand',
 'Capacity_OZ_',
 'CarpetAndRugInstituteCertification',
 'CenterforResourceSolutionsGreen_eCertification',
 'CompostingManufacturingAlliance_CMA_Certification',
 'EPADesignforEnvironment_DfE_Certification',
 'EPASaferChoiceCertification',
 'ForestStewardshipCouncil_FSC_Certification',
 'GreenRestaurantAssociationCertification',
 'GreenSealProductCertification',
 'MaterialThickness_MIL_',
 'NationalSanitationFoundation_NSF_Certification',
 'PrimaryImage',
 'ProductCapa

In [ ]:
# transactions = pd.read_excel(f"{ip_path_cat}Cups Sales Data L6M Feb-Jul.xlsx")
# item_master = pd.read_csv(f"{ip_path}consolidated_item_master_by_location_20250610152008.csv")

In [10]:
# Reading Sales Data
sales_data_2025 = pd.read_csv("C:/Users/MGadupudi/Desktop/Sales_Database_2025.csv",
                               index_col=False,
                               encoding='windows-1252',
                               dtype=str)


sales_data_2025

,Fiscal Month,Warehouse No.,Branch Location,Entity Code,ERP,Customer Code,Ship to Code,Price Group Name,Customer Class,Item Code,...,Burden Cost,Qty,POD Rebate,Gross Profit,Gross Sales (converted),Gross Cost (converted),Net Cost (converted),Burden Cost (converted),POD Rebate (converted),Gross Profit (converted)
0,201205,12--4,"Austin, TX",12,Gulf Coast Infor A+,8009299,5,"Charter Communications, Inc",Corporate,6310,...,2.28,2,0,1.08,NaN,NaN,NaN,NaN,NaN,NaN
1,201205,12--4,"Austin, TX",12,Gulf Coast Infor A+,8009299,5,"Charter Communications, Inc",Corporate,6311,...,3.58,2,0,1.38,NaN,NaN,NaN,NaN,NaN,NaN
2,201205,12--4,"Austin, TX",12,Gulf Coast Infor A+,8009299,5,"Charter Communications, Inc",Corporate,GP16880,...,129.785,2,9.90000000000001,27.4,NaN,NaN,NaN,NaN,NaN,NaN
3,201205,12--4,"Austin, TX",12,Gulf Coast Infor A+,8009299,5,"Charter Communications, Inc",Corporate,MB540A,...,127.9872,6,31.2,76.98,NaN,NaN,NaN,NaN,NaN,NaN
4,201205,12--4,"Austin, TX",12,Gulf Coast Infor A+,8009299,5,"Charter Communications, Inc",Corporate,R2433N8,...,26.5125,1,-0.579999999999998,7.8,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19047530,202511,92--10,"Landover, MD",92,SFS P21,725394,975291,Absolute Service Industries,Facility Services,998017535,...,NaN,3,15.3299997,24.21,108.87,99.9899997,84.66,NaN,15.3299997,24.21
19047531,202511,92--10,"Landover, MD",92,SFS P21,725394,975291,Absolute Service Industries,Facility Services,998052123,...,NaN,1,0,17.0144,89.16,72.1456,72.1456,NaN,0,17.0144
19047532,202512,12--4,"Austin, TX",12,Gulf Coast Infor A+,8011705,8,Mccoy's,Corporate,GP19375,...,104.737,2,24.2,23.32,127.02,127.9,103.7,104.737,24.2,23.32
19047533,202512,12--4,"Austin, TX",12,Gulf Coast Infor A+,8011705,8,Mccoy's,Corporate,GP42715,...,35.249,1,0.550000000000004,10.41,45.31,35.45,34.9,35.249,0.550000000000004,10.41


In [11]:
sales_data_2025['Fiscal Month'].unique()

array(['201205', '201601', '202409', '202412', '202501', '202502',
       '202503', '202504', '202505', '202506', '202507', '202508',
       '202509', '202510', '202511', '202512'], dtype=object)

In [13]:
item_df_real_time = pd.read_csv('Item_Master_real_time.csv')

C:\Users\MGadupudi\AppData\Local\Temp\ipykernel_58472\1491028614.py:1: DtypeWarning: Columns (1,2,3,4,5,7,8,9,11,12,13,14,15,16,17,18,19,20,21,22,24,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,53,54,55,57,59,60,61,62,63,65,66,67,68,69,70,71,72,73,74) have mixed types. Specify dtype option on import or set low_memory=False.
  item_df_real_time = pd.read_csv('Item_Master_real_time.csv')


In [14]:
item_segment_mapping = pd.read_csv("C:/Users/MGadupudi/Desktop/Item Segment Mapping.csv", index_col=False,
                               encoding='windows-1252',
                               dtype=str)
# Reading Item Master
conn_str = (
    r'DRIVER=SQL Server;'
    r'SERVER=ibp-db01;'
    r'DATABASE=fornax;'
)

conn = pyodbc.connect(conn_str)
sql_query = "select * from dbo.ConsolidatedItemsByCompany"
item_df_real_time = pd.read_sql(sql_query, conn)
conn.close()

In [15]:
item_segment_mapping.head()

,Parent Category,Parent Category Description,Division,Div Description,Class,Class Description,Item Division,Item Segment Key,Item Sub Category,Item Category,...,Duplicates Check,Entity,Entity Name,V1,V2,V3,IC CIM,V1A,V2A,Duplicates Check 2
0,A1,Food Service Disposables,Z1,Grocery & Shopping Bags,NaN,NaN,Food Service Disposables-Grocery & Shopping Bags,21-A1-Z1-,Bags,Bags,...,1,21,APP,NaN,NaN,NaN,FSP,-,-,1
1,A1,Food Service Disposables,Z2,Plastic Bread & Poly Bags,NaN,NaN,Food Service Disposables-Plastic Bread & Poly ...,21-A1-Z2-,Bags,Bags,...,1,21,APP,NaN,NaN,NaN,FSP,-,-,1
2,A2,Jan/San,Z3,Bags - Autoclave,NaN,NaN,Jan/San-Bags - Autoclave,21-A2-Z3-,Bags,Bags,...,1,21,APP,NaN,NaN,NaN,FSP,-,-,1
3,A1,Food Service Disposables,Z4,Deli & Produce Bags,NaN,NaN,Food Service Disposables-Deli & Produce Bags,21-A1-Z4-,Bags,Bags,...,1,21,APP,NaN,NaN,NaN,FSP,-,-,1
4,A2,Jan/San,Z5,Compostable,NaN,NaN,Jan/San-Compostable,21-A2-Z5-,Janitorial Supplies,Janitorial Supplies,...,1,21,APP,NaN,NaN,NaN,Jan San,-,-,1


In [16]:
item_segment_mapping.columns

Index(['Parent Category', 'Parent Category Description', 'Division',
       'Div Description', 'Class', 'Class Description', 'Item Division',
       'Item Segment Key', 'Item Sub Category', 'Item Category',
       'Item Segment', 'Commodity', 'COVID-19 VIEW', 'CIM Category',
       'Simple CIM Category', 'Duplicates Check', 'Entity', 'Entity Name',
       'V1', 'V2', 'V3', 'IC CIM', 'V1A', 'V2A', 'Duplicates Check 2'],
      dtype='object')

In [17]:
item_df_real_time.head()

,Company No.,Location,Region,Warehouse Code,Warehouse Name,Item Code,Item Description 1,Item Description 2,Item Description 3,Purchasing UoM,...,tibco_flag,ERP_Status,Time_Zone,Daylight_Savings_Flag,Entity_Name,Warehouse_Name,Entity_Name_Old,Entity_Name_New,Whs_Name_Old,Whs_Name_New
0,1,Imperial,NaN,NaN,NaN,62323PFS,VB GLOVES LATEX SM IVORY GP,PF FOOD CONTACT 10/100,B/C 62323PFSB - 62323PFS,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Imperial,NaN,NaN,NaN,FT13558,FOOD TOWN PRODUCE BAG,15X20 7.25MIC 4RLS/750 PNP,FT13558,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Imperial,NaN,NaN,NaN,VBPK16S,VB CONT BASE DELI FOOD 16 OZ,CLR RND PP 10/50 CS,VBPK16S VBPVBPK16S,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,Imperial,NaN,NaN,NaN,VBPK8S,VB CONT BASE DELI FOOD 8 OZ,CLR RND PP 10/50 CS,VBPPLID VBPK8S,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,Imperial,NaN,NaN,NaN,VBPPLID,VB LID CLR RND PLUG FIT FOR,6/8/12/16/24/32 OZ DELI CONT,. VBPPLID,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
item_df_real_time.columns

Index(['Company No.', 'Location', 'Region', 'Warehouse Code', 'Warehouse Name',
       'Item Code', 'Item Description 1', 'Item Description 2',
       'Item Description 3', 'Purchasing UoM', 'VGN Code', 'VGN',
       'Preferred Vendor Code', 'Preferred Vendor name', 'VPN', 'Stock',
       'Status', 'Item Category Code', 'Item Category Name',
       'Item Sub Category Code', 'Item Sub Category Name', 'PO Cost',
       'Loaded Cost', 'Qty On Hand', 'Case Pack', 'Cube', 'Weight',
       'Private Label Flag', 'Last Received Date', 'PO Cost Effective date',
       'Item Master Report date', 'Private_Label_Name', 'Territory',
       'ERP_System', 'WHS_Status', 'Company_Name', 'Stock_Group_Name',
       'Status_Group_Name', 'Printed Item flag', 'Last Sale Date', 'UPC 12',
       'UPC 14 / GTIN', 'Manufacturer Code', 'Manufacturer Name',
       'Manufacturer Item Code', 'Hazard Code', 'Avg Cost',
       'ERP System Instance', 'Company_UOM', 'Item Parent Category Code',
       'Item Parent Cate

In [19]:
item_segment_mapping['Item Segment Key'] = item_segment_mapping['Item Segment Key'].astype(str).str.strip()
item_df_real_time['Item Segment Key'] = item_df_real_time['Item Segment Key'].astype(str).str.strip()

item_df_real_time = item_df_real_time.merge(
    item_segment_mapping[['Item Segment Key', 'Item Sub Category']],
    on='Item Segment Key',
    how='left'
)

In [20]:
item_df_real_time.head()

,Company No.,Location,Region,Warehouse Code,Warehouse Name,Item Code,Item Description 1,Item Description 2,Item Description 3,Purchasing UoM,...,ERP_Status,Time_Zone,Daylight_Savings_Flag,Entity_Name,Warehouse_Name,Entity_Name_Old,Entity_Name_New,Whs_Name_Old,Whs_Name_New,Item Sub Category
0,1,Imperial,NaN,NaN,NaN,62323PFS,VB GLOVES LATEX SM IVORY GP,PF FOOD CONTACT 10/100,B/C 62323PFSB - 62323PFS,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gloves
1,1,Imperial,NaN,NaN,NaN,FT13558,FOOD TOWN PRODUCE BAG,15X20 7.25MIC 4RLS/750 PNP,FT13558,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Printed Merch
2,1,Imperial,NaN,NaN,NaN,VBPK16S,VB CONT BASE DELI FOOD 16 OZ,CLR RND PP 10/50 CS,VBPK16S VBPVBPK16S,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Containers
3,1,Imperial,NaN,NaN,NaN,VBPK8S,VB CONT BASE DELI FOOD 8 OZ,CLR RND PP 10/50 CS,VBPPLID VBPK8S,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Containers
4,1,Imperial,NaN,NaN,NaN,VBPPLID,VB LID CLR RND PLUG FIT FOR,6/8/12/16/24/32 OZ DELI CONT,. VBPPLID,CS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Containers


In [ ]:
# item_df_real_time.to_csv('Item_Master_real_time.csv', index=False)

In [ ]:
sales_data_2025.columns

In [21]:
# Convert Fiscal Month from 'YYYYMM' format to datetime
# Format '201205' -> datetime(2012, 5, 1)
sales_data_2025['Fiscal Month Date'] = pd.to_datetime(sales_data_2025['Fiscal Month'], format='%Y%m', errors='coerce')

In [22]:
# Calculate the cutoff date (6 months ago from today)
six_months_ago = datetime.now() - timedelta(days=220)

In [23]:
six_months_ago

datetime.datetime(2025, 6, 30, 15, 18, 20, 77201)

In [24]:
# Filter for last 6 months from current date
sales_data_filtered = sales_data_2025[sales_data_2025['Fiscal Month Date'] >= six_months_ago].copy()

In [26]:
item_df_real_time[(item_df_real_time['Company No.'] == 1) & (item_df_real_time['Item Code'] == 'KON8DW')]

,Company No.,Location,Region,Warehouse Code,Warehouse Name,Item Code,Item Description 1,Item Description 2,Item Description 3,Purchasing UoM,...,ERP_Status,Time_Zone,Daylight_Savings_Flag,Entity_Name,Warehouse_Name,Entity_Name_Old,Entity_Name_New,Whs_Name_Old,Whs_Name_New,Item Sub Category
1158426,1,Jersey City,Northeast,JC,Jersey City,KON8DW,KONDITONI 8 OZ DOUBLE WALL,PTD,KON8DW KON8DW,CS,...,Active,EST,Y,Imperial Dade,Jersey City,Imperial Dade,Imperial Dade,Jersey City,Jersey City,Printed Merch
1975755,1,Bordentown,Northeast,NJBT,Bordentown,KON8DW,KONDITONI 8 OZ DOUBLE WALL,PTD,KON8DW KON8DW,CS,...,Active,EST,Y,Imperial Dade,Bordentown,Imperial Dade,Imperial Dade,Bordentown,Bordentown,Printed Merch


In [27]:
sales_data_filtered.head()

,Fiscal Month,Warehouse No.,Branch Location,Entity Code,ERP,Customer Code,Ship to Code,Price Group Name,Customer Class,Item Code,...,Qty,POD Rebate,Gross Profit,Gross Sales (converted),Gross Cost (converted),Net Cost (converted),Burden Cost (converted),POD Rebate (converted),Gross Profit (converted),Fiscal Month Date
11204996,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,18X2M,...,1,1.34,4.6,16.29,13.03,11.69,14.7434,1.34,4.6,2025-07-01
11204997,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,20100F,...,1,0,4.64,17.65,13.01,13.01,18.73,0,4.64,2025-07-01
11204998,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,20102,...,2,0,-5.5,88.6,94.1,94.1,100.1,0,-5.5,2025-07-01
11204999,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,2025NAT65188,...,1,0,3.37,38.78,35.41,35.41,38.81,0,3.37,2025-07-01
11205000,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,24X2M,...,2,2.54,10.88,41.7,33.36,30.82,38.1768,2.54,10.88,2025-07-01


In [28]:
item_df_real_time['Warehouse No.'] = item_df_real_time['Company No.'].astype(str).str.strip() + '--' + item_df_real_time['Warehouse Code'].astype(str).str.strip()

In [29]:
item_df_real_time['Warehouse No.'].tail()

8213425    100--SKSA
8213426    100--SKSA
8213427    100--SKSA
8213428    100--SKSA
8213429    100--SKSA
Name: Warehouse No., dtype: object

In [30]:
# Merge sales data with item data
# Using Item Code and Warehouse No. as the join keys
sales_data_filtered['Warehouse No.'] = sales_data_filtered['Warehouse No.'].str.strip()
transactions = sales_data_filtered.merge(
    item_df_real_time,
    left_on=['Item Code', 'Warehouse No.'],
    right_on=['Item Code', 'Warehouse No.'],
    how='inner'
)

In [31]:
sales_data_filtered.shape

(7842539, 33)

In [32]:
transactions.shape

(6265261, 108)

In [33]:
transactions.head()

,Fiscal Month,Warehouse No.,Branch Location,Entity Code,ERP,Customer Code,Ship to Code,Price Group Name,Customer Class,Item Code,...,ERP_Status,Time_Zone,Daylight_Savings_Flag,Entity_Name,Warehouse_Name,Entity_Name_Old,Entity_Name_New,Whs_Name_Old,Whs_Name_New,Item Sub Category
0,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,18X2M,...,Active,PST,Y,Imperial Dade,City of Industry,American Paper & Plastics,Imperial Dade,City of Industry,City Of Industry,Films & wraps
1,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,20100F,...,Active,PST,Y,Imperial Dade,City of Industry,American Paper & Plastics,Imperial Dade,City of Industry,City Of Industry,Equipment
2,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,20102,...,Active,PST,Y,Imperial Dade,City of Industry,American Paper & Plastics,Imperial Dade,City of Industry,City Of Industry,Catering
3,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,2025NAT65188,...,Active,PST,Y,Imperial Dade,City of Industry,American Paper & Plastics,Imperial Dade,City of Industry,City Of Industry,Bags
4,202507,1--CACI,"City of Industry, CA",1,Imperial S2K,ALT068,0001,Alonti Catering Kitchens,Caterer,24X2M,...,Active,PST,Y,Imperial Dade,City of Industry,American Paper & Plastics,Imperial Dade,City of Industry,City Of Industry,Films & wraps


In [34]:
sales_data_filtered.columns

Index(['Fiscal Month', 'Warehouse No.', 'Branch Location', 'Entity Code',
       'ERP', 'Customer Code', 'Ship to Code', 'Price Group Name',
       'Customer Class', 'Item Code', 'Drop Ship', 'sales_rep_id', 'Zip Code',
       'State', 'City', 'Native Currency', 'Conv Rate (Aligned FOREX)',
       'Conv Rate (AVG)', 'Conv Rate (EOM)', 'Gross Sales', 'Gross Cost',
       'Net Cost', 'Burden Cost', 'Qty', 'POD Rebate', 'Gross Profit',
       'Gross Sales (converted)', 'Gross Cost (converted)',
       'Net Cost (converted)', 'Burden Cost (converted)',
       'POD Rebate (converted)', 'Gross Profit (converted)',
       'Fiscal Month Date'],
      dtype='object')

In [35]:
item_df_real_time.columns

Index(['Company No.', 'Location', 'Region', 'Warehouse Code', 'Warehouse Name',
       'Item Code', 'Item Description 1', 'Item Description 2',
       'Item Description 3', 'Purchasing UoM', 'VGN Code', 'VGN',
       'Preferred Vendor Code', 'Preferred Vendor name', 'VPN', 'Stock',
       'Status', 'Item Category Code', 'Item Category Name',
       'Item Sub Category Code', 'Item Sub Category Name', 'PO Cost',
       'Loaded Cost', 'Qty On Hand', 'Case Pack', 'Cube', 'Weight',
       'Private Label Flag', 'Last Received Date', 'PO Cost Effective date',
       'Item Master Report date', 'Private_Label_Name', 'Territory',
       'ERP_System', 'WHS_Status', 'Company_Name', 'Stock_Group_Name',
       'Status_Group_Name', 'Printed Item flag', 'Last Sale Date', 'UPC 12',
       'UPC 14 / GTIN', 'Manufacturer Code', 'Manufacturer Name',
       'Manufacturer Item Code', 'Hazard Code', 'Avg Cost',
       'ERP System Instance', 'Company_UOM', 'Item Parent Category Code',
       'Item Parent Cate

In [36]:
transactions.shape

(6265261, 108)

In [37]:
transactions = transactions[(transactions['Item Sub Category'] == 'Cups') & (transactions['Entity Code'] == '1')]

In [38]:
transactions.shape

(250662, 108)

In [39]:
transactions.columns

Index(['Fiscal Month', 'Warehouse No.', 'Branch Location', 'Entity Code',
       'ERP', 'Customer Code', 'Ship to Code', 'Price Group Name',
       'Customer Class', 'Item Code',
       ...
       'ERP_Status', 'Time_Zone', 'Daylight_Savings_Flag', 'Entity_Name',
       'Warehouse_Name', 'Entity_Name_Old', 'Entity_Name_New', 'Whs_Name_Old',
       'Whs_Name_New', 'Item Sub Category'],
      dtype='object', length=108)

In [40]:
# Create the final transactions table with the exact columns you need
transactions_final = pd.DataFrame({
    'Entity Code': transactions['Entity Code'],
    'Region': transactions['Region'],
    'Territory': transactions['Territory'],
    'Location PS': transactions['Location'],
    'Dashboard Location': transactions['Dashboard_Location'],
    'Warehouse': transactions['Warehouse Name'],
    'Whs Code': transactions['Warehouse Code'],
    'ERP System': transactions['ERP_System'],
    'Item': transactions['Item Code'],
    'Item Desc 1': transactions['Item Description 1'],
    'Item Desc 2': transactions['Item Description 2'],
    'Item Category': transactions['Item Category Name'],
    'Item Sub Category': transactions['Item Sub Category'],
    'Class Description': transactions['Customer Class'],
    'VPN': transactions['VPN'],
    'Selling UOM': transactions['Company_UOM'],
    'IM UoM': transactions['Purchasing UoM'],
    'Case Pack': transactions['Case Pack'],
    'Updated Salesperson Name': transactions['sales_rep_id'],
    'Customer Code': transactions['Customer Code'],
    'Price Group Name': transactions['Price Group Name'],
    'Customer Class': transactions['Customer Class'],
    'Preferred Vendor Code': transactions['Preferred Vendor Code'],
    'Preferred Vendor Name': transactions['Preferred Vendor name'],
    'VGN': transactions['VGN'],
    'Qty': transactions['Qty'],
    'Gross Cost': transactions['Gross Cost'],
    'Gross Sales': transactions['Gross Sales'],
    'Gross Sales Converted': transactions['Gross Sales (converted)'],
    'Net Cost': transactions['Net Cost'],
    'POD': transactions['POD Rebate'],
    'VB Flag': transactions['VB_Flag'],
    'Printed Item Flag': transactions['Printed Item flag'],
})

In [41]:
transactions_final.head()

,Entity Code,Region,Territory,Location PS,Dashboard Location,Warehouse,Whs Code,ERP System,Item,Item Desc 1,...,Preferred Vendor Name,VGN,Qty,Gross Cost,Gross Sales,Gross Sales Converted,Net Cost,POD,VB Flag,Printed Item Flag
29,1,West,West,APP,"City of Industry, CA",City Of Industry,CACI,S2K,VBCLLH16DB,VB LID HOT CUP DOME BLK 92MM,...,GRAPHIC PACKAGING,Graphic Packaging,1,23.94,20.03,20.03,23.94,0,Y - VB,N
108,1,West,West,APP,"City of Industry, CA",City Of Industry,CACI,S2K,VBCLLH16DB,VB LID HOT CUP DOME BLK 92MM,...,GRAPHIC PACKAGING,Graphic Packaging,1,17.22,20.03,20.03,17.22,0,Y - VB,N
145,1,West,West,APP,"City of Industry, CA",City Of Industry,CACI,S2K,PTC09D92,VB CUP COLD 9 OZ SQUAT PET,...,CARRYOUT BAGS -PET CUPS / HIPS,"Carryout Bags, Inc.",1,41.4,31.39,31.39,22.55,18.85,Y - VB,N
176,1,West,West,APP,"City of Industry, CA",City Of Industry,CACI,S2K,CPLUGBLACK,PLUG CIRCLE BLK FOR HOT CUP,...,"AMERCAREROYAL, LLC",Amercareroyal,17,428.74,878.9,878.9,428.74,0,N,N
211,1,West,West,APP,"City of Industry, CA",City Of Industry,CACI,S2K,2CUPCARRY,CARRIER HOLDER FOR 2 CUP,...,CARRYOUT BAGS - #1003,"Carryout Bags, Inc.",2,70,108.12,108.12,70,0,Y - Other,N


In [42]:
transactions_final['Printed Item Flag'].unique()

array(['N'], dtype=object)

In [43]:
item_df_real_time['Printed Item flag'].unique()

array(['N', 'Y', nan], dtype=object)

In [44]:
# Rename to item_master and standardize column names
item_master = item_df_real_time.rename(columns={
    'Company No.': 'branch_code',
    'Location': 'city_txt',
    'Region': 'region_code',
    'Warehouse Code': 'location_code',
    'Warehouse Name': 'location_name',
    'Item Code': 'item_code',
    'Item Description 1': 'description_line1_txt',
    'Item Description 2': 'description_line2_txt',
    'Item Description 3': 'description_line3_txt',
    'Purchasing UoM': 'uom_purchase_type',
    'VGN Code': 'vgn_code',
    'VGN': 'vgn_name',
    'Preferred Vendor Code': 'vendor_code',
    'Preferred Vendor name': 'vendor_name',
    'VPN': 'vpn_code',
    'Stock': 'stock_item_flag',
    'Status': 'status_type',
    'Item Category Code': 'item_category_level2_code',
    'Item Category Name': 'item_category_level2_name',
    'Item Sub Category Code': 'item_category_level3_code',
    'Item Sub Category Name': 'item_category_level3_name',
    'PO Cost': 'po_cost_amt',
    'Loaded Cost': 'loaded_cost_amt',
    'Qty On Hand': 'onhand_qty',
    'Case Pack': 'unit_case_pack_desc',
    'Cube': 'cube_val',
    'Weight': 'weight_val',
    'Private Label Flag': 'private_label_flag',
    'Last Received Date': 'last_received_dt',
    'PO Cost Effective date': 'effective_po_cost_dt',
    'Item Master Report date': 'report_dt',
    'Private_Label_Name': 'private_label_name',
    'Territory': 'territory_name',
    'ERP_System': 'erp_system_name',
    'WHS_Status': 'location_status_type',
    'Company_Name': 'branch_name',
    'Stock_Group_Name': 'stock_group_type',
    'Status_Group_Name': 'status_group_type',
    'Printed Item flag': 'printed_item_flag',
    'Last Sale Date': 'last_sale_dt',
    'UPC 12': 'upc_12_code',
    'UPC 14 / GTIN': 'upc_14_code',
    'Manufacturer Code': 'manufacturer_code',
    'Manufacturer Name': 'manufacturer_name',
    'Manufacturer Item Code': 'manufacturer_item_code',
    'Hazard Code': 'hazard_code',
    'Avg Cost': 'average_cost_amt',
    'Item Parent Category Code': 'item_category_level1_code',
    'Item Parent Category Name': 'item_category_level1_name',
    'Item Segment Key': 'item_segment_key',
    'L4M_Usage_Qty': 'l4m_usage_qty',
    'UOM_Group_Name': 'uom_group_type',
    'VB_Flag': 'victoriabay_code',
    'Item Sub Category': 'item_sub_category'
})

In [45]:
transactions_final['Gross Sales'] = transactions_final['Gross Sales'].astype(float)
test_grouped = transactions_final.groupby(['Item'])['Gross Sales'].sum().sort_values(ascending=False)

In [46]:
test_grouped

Item
VG16CF       2.985866e+06
VBCLHP12W    1.798010e+06
F98HCF       1.316872e+06
VG12CF       9.485554e+05
16J16        9.219159e+05
                 ...     
EPDLCCNH    -1.634233e+02
PCC16PEP    -4.378800e+02
RP12SP      -4.818900e+02
YPP302N     -1.741040e+03
NC24        -2.446659e+03
Name: Gross Sales, Length: 1728, dtype: float64

In [47]:
transactions_final['Entity--Item'] = transactions_final['Entity Code'].str.strip() + '--' + transactions_final['Item'].str.strip().str.upper()

item_master['Entity--Item'] = item_master['branch_code'].astype(str).str.strip() + '--' + item_master['item_code'].str.strip().str.upper()

In [48]:
sfy['Entity--Item'] = '1--'+sfy['S2K Item Number'].astype(str).str.strip().str.upper()
# transactions = transactions[transactions['Item Sub Category'] == 'Cups']

In [49]:
item_master.shape

(8213430, 78)

In [50]:
item_master = item_master[item_master['Entity--Item'].isin(transactions_final['Entity--Item'])]

In [51]:
transactions_final['Qty'] = transactions_final['Qty'].astype(float).astype(int)
transactions_final['Net Cost'] = transactions_final['Net Cost'].astype(float).astype(int)
transactions_final['Gross Cost'] = transactions_final['Gross Cost'].astype(float).astype(int)


In [52]:
transactions_added = lif.add_po_cost(transactions_final, item_master)

  Updated 31 transactions with Qty = Net Cost / PO Cost
  Updated 102 transactions with Gross Cost = PO Cost × Qty
PO Cost matching complete:
  Total transactions: 250662
  Matched transactions: 250662
  Match rate: 100.0%
  Total data fixes applied: 133 (Qty: 31, Gross Cost: 102, Net Cost: 0, PO Cost: 0)
  Data cleanup complete:
    Removed 5997 rows with negative values
    Final row count: 244665 (from 250662)


In [53]:
im_grp = lif.group_data(transactions_added)

C:\Users\MGadupudi\PycharmProjects\ImperialDadeCategoryManagement\Function_Files\Load_Isolate_functions.py:126: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  po_cost_weighted = im_grp.groupby(group_cols).apply(


In [ ]:
# transactions_added = pd.read_csv(f"{OP_PATH}Post DB Addition Cups Sales Data L6M Feb-Jul - Cleaned.csv")

In [54]:
transactions_added.to_csv(f"{OP_PATH}Post DB Addition Cups Sales Data L6M Feb-Jul - Cleaned.csv", index=False)

In [55]:
im_s2k, columns_with_coverage, example_data = lif.get_columns_with_coverage(im_grp, sfy, 15)

Filtered sfy to 105231 rows
Found 6 columns meeting 15% coverage
Extracted sample data for 6 columns


In [95]:
columns_with_coverage

['Product Type Collapse',
 'Pack Size',
 'Color',
 'Material',
 'Product Attributes',
 'Product Dimension Type']

In [56]:
columns_with_coverage

['Product Type Collapse',
 'Pack Size',
 'Color',
 'Material',
 'Product Attributes',
 'Product Dimension Type']

In [57]:
sorted(list(sfy.columns))

['Active Location Count',
 'Additional Images',
 'Air Freshener & Deodorizer Product Type',
 'Air Purifier Product Type',
 'Aluminum Pan Rim Style',
 'Assembly Required',
 'Associated Product ID',
 'Baby Care Product Type',
 'Baby Changing Table & Liner Product Type',
 'Backing Material',
 'Bag & Box Closure Style',
 'Bag Length (IN)',
 'Bag Lip (IN)',
 'Bag Width (IN)',
 'Bakery Supply Product Features',
 'Bakery Supply Product Type',
 'Bakery Supply Product Usage',
 'Batteries Size',
 'Bed & Bath Linen Product Type',
 'Beverage Cup Style',
 'Beverage Cup Type',
 'Biodegradable Products Institute (BPI) Certification',
 'Bottle Mouth Style',
 'Bottom Diameter (IN)',
 'Bottom Width (IN)',
 'Box Flute Size',
 'Brand',
 'Brand Logo',
 'Brand Reference',
 'Branded Product Name',
 'Broom Product Type',
 'Broom Product Usage',
 'Brush Bristle Material',
 'Brush Product Type',
 'Bucket & Wringer Product Type',
 'Building & Facility Maintenance Product Features',
 'Building & Facility Maintena

In [58]:
# Can manually check/change the columns with coverage to see if they are correct
columns_for_description = [
'Beverage Cup Type', - Critical (Is it a cup or lid make the cups match with cups and lids with lids or combo)
 'Product Capacity', - SAME
 'Usage Temperature', - SAME
 'Material', - Can be differnt

 'Product Type Collapse',
 'Pack Size',
 'Color',
 'Foodservice Global Attributes',
 'Beverage Cup Style',
 'Sustainable Products',
 'Compatible Product & Product Type',
 'Bottom Diameter (IN)',
 'Top Diameter (IN)',
 'Product Dimension Type',
 'Product Dimensions',
 'Pattern & Design'
]

#### Merge data with salsify

In [59]:
# get columns with coverage from sfy dataframe and merge with im_concat dataframe
sfy_covered = sfy[columns_for_description+['Entity--Item']].copy()
im_final = im_grp.merge(sfy_covered, on='Entity--Item', how='left', suffixes=('', '_dup'))

In [60]:
im_final[im_final.duplicated(subset=['Entity--Item'], keep=False)]
# keep first and set columns_with_coverage to nan
im_final = im_final.drop_duplicates(subset=['Entity--Item'], keep='first')
for col in columns_with_coverage:
    if col in im_final.columns:
        im_final[col] = im_final[col].fillna('')

In [61]:
im_final

,Entity--Item,Item Desc 1,Item Desc 2,Qty,Gross Cost,Net Cost,po_cost_amt,Case Pack,VB Flag,VGN,...,Usage Temperature,Beverage Cup Style,Sustainable Products,Product Capacity,Compatible Product & Product Type,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Product Dimensions,Pattern & Design
0,1--VBCLLH16DB,VB LID HOT CUP DOME BLK 92MM,PS 10 OZ SQUAT-24 OZ 20/50,10474,185100,185880,20.629195,1000,Y - VB,Graphic Packaging,...,Hot Only,NaN,Yes,16 OZ,NaN,NaN,NaN,Standard Product Dimensions,NaN,NaN
1,1--PTC09D92,VB CUP COLD 9 OZ SQUAT PET,92 SERIES CLR 20/50,15365,487038,339648,43.747324,1000,Y - VB,"Carryout Bags, Inc.",...,Cold Only,NaN,NaN,9 OZ,NaN,2.4,3.6,Tapered & Graduated Product Dimensions,3.6X2.8X2.4 IN,NaN
2,1--CPLUGBLACK,PLUG CIRCLE BLK FOR HOT CUP,LID SIP HOLE 5/400 BLACK,1507,37845,37845,24.190000,2000,N,Amercareroyal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN
3,1--2CUPCARRY,CARRIER HOLDER FOR 2 CUP,MOLDED FIBER 8-24 OZ,247,8645,8645,36.050000,600.0,Y - Other,"Carryout Bags, Inc.",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN
4,1--DLKC12/20NH,LID CUP COLD DOME CLR PET,W/ NO HOLE KAL-CLR,412,27053,15549,70.922743,1000,N,Pactiv,...,Cold Only,NaN,NaN,NaN,Cup,NaN,NaN,Standard Product Dimensions,3.8X1.6 IN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1650,1--N8124,8 OZ. 1 PC. WINE GLASSES BOX,SET CLEAR N8124,5,109,109,24.700000,48.0,N,North West Enterprises,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN
1651,1--N824099,8OZ WINE STEM CLR 1PC,24/10CS DELUX N824099,5,342,342,76.890000,240.0,N,North West Enterprises,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN
1652,1--WINE52020,5.5OZ WINE GLASS CLEAR 2PC,20/20CS WINE5-20/20,5,281,281,63.290000,400.0,N,North West Enterprises,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN
1653,1--DXE5356DX,16 OZ PERFECT TOUCH HOT CUP,500/CS,4,332,332,83.050000,500,N,Essendant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN


#### Write files - this is the subsection of to_keep that fits the category

In [62]:
OP_PATH = f"{parent_dir}\\Data\\{CATEGORY}\\Output\\"
os.makedirs(OP_PATH, exist_ok=True)  
im_final.to_csv(f"{OP_PATH}{CATEGORY}_SKUS_with_Salsify.csv", index=False)

In [63]:
im_final = pd.read_csv(f"{OP_PATH}{CATEGORY}_SKUS_with_Salsify.csv")

In [64]:
im_final.head()

,Entity--Item,Item Desc 1,Item Desc 2,Qty,Gross Cost,Net Cost,po_cost_amt,Case Pack,VB Flag,VGN,...,Usage Temperature,Beverage Cup Style,Sustainable Products,Product Capacity,Compatible Product & Product Type,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Product Dimensions,Pattern & Design
0,1--VBCLLH16DB,VB LID HOT CUP DOME BLK 92MM,PS 10 OZ SQUAT-24 OZ 20/50,10474,185100,185880,20.629195,1000.0,Y - VB,Graphic Packaging,...,Hot Only,NaN,Yes,16 OZ,NaN,NaN,NaN,Standard Product Dimensions,NaN,NaN
1,1--PTC09D92,VB CUP COLD 9 OZ SQUAT PET,92 SERIES CLR 20/50,15365,487038,339648,43.747324,1000.0,Y - VB,"Carryout Bags, Inc.",...,Cold Only,NaN,NaN,9 OZ,NaN,2.4,3.6,Tapered & Graduated Product Dimensions,3.6X2.8X2.4 IN,NaN
2,1--CPLUGBLACK,PLUG CIRCLE BLK FOR HOT CUP,LID SIP HOLE 5/400 BLACK,1507,37845,37845,24.190000,2000.0,N,Amercareroyal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1--2CUPCARRY,CARRIER HOLDER FOR 2 CUP,MOLDED FIBER 8-24 OZ,247,8645,8645,36.050000,600.0,Y - Other,"Carryout Bags, Inc.",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1--DLKC12/20NH,LID CUP COLD DOME CLR PET,W/ NO HOLE KAL-CLR,412,27053,15549,70.922743,1000.0,N,Pactiv,...,Cold Only,NaN,NaN,NaN,Cup,NaN,NaN,Standard Product Dimensions,3.8X1.6 IN,NaN


In [65]:
im_final[im_final['Entity--Item'] == '1--KON8DW']

,Entity--Item,Item Desc 1,Item Desc 2,Qty,Gross Cost,Net Cost,po_cost_amt,Case Pack,VB Flag,VGN,...,Usage Temperature,Beverage Cup Style,Sustainable Products,Product Capacity,Compatible Product & Product Type,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Product Dimensions,Pattern & Design


## Classification

#### Generate parts for taxonomy prompt

In [66]:
# generate a string of the top 5 values for each column in columns_for_description to help with the prompt
prompt_options_string = cf.get_top_values(im_final, columns_for_description, 5)
output_str = cf.get_most_common_values(prompt_options_string)

In [67]:
import numpy as np
im_final = im_final.replace("", np.nan)

In [68]:
# create a description string for each row in that will be used in the promp
im_final['Combined Descriptions'] = (
    im_final['Item Desc 1'].fillna('') + ' ' +
    im_final['Item Desc 2'].fillna('') + ' ' #+
    #im_final['description_line3_txt'].fillna('')
    ).str.strip()

existing_columns_for_description = [col for col in columns_for_description if col in im_final.columns]

if not existing_columns_for_description:
    print("Warning: None of the specified columns for description exist in the DataFrame.")
    im_final['Description with Attributes'] = "" # Create an empty description column
else:
    if len(existing_columns_for_description) < len(columns_for_description):
        missing_cols = set(columns_for_description) - set(existing_columns_for_description)
        print(f"Warning: The following specified columns were explain_top_qty_descriptionnot found in the DataFrame and will be skipped: {missing_cols}")
    
im_final['Description with Attributes'] = im_final.apply(
    lambda row: cf.create_description_string(row, existing_columns_for_description),
    axis=1 # Apply function row-wise
)

In [69]:
example_desc, example_output = cf.explain_top_qty_description(im_final, output_str)

Running cost $0.0000: 100%|██████████| 1/1 [00:02<00:00,  2.29s/chunk]


In [70]:
example_desc

'Beverage Cup Type: Cup,  Product Type Collapse: Cup,  Pack Size: 1000/Case,  Color: Clear,  Material: Polyethylene Terephthalate (PET),  Foodservice Global Attributes: Disposable,  Usage Temperature: Cold Only,  Product Capacity: 16 OZ,  Bottom Diameter (IN): 2.5,  Top Diameter (IN): 3.9,  Product Dimension Type: Tapered & Graduated Product Dimensions,  Product Dimensions: 3.9X4.7X2.5 IN, VB CUP COLD 16 OZ PET CLR 98 MM SERIES 20/50'

In [71]:
example_output

'Beverage Cup Type: Cup | Product Type Collapse: Cup | Pack Size: 1000/Case | Color: Clear | Material: Polyethylene Terephthalate (PET) | Foodservice Global Attributes: Disposable | Usage Temperature: Cold Only | Beverage Cup Style:  | Sustainable Products:  | Product Capacity: 16 OZ | Compatible Product & Product Type:  | Bottom Diameter (IN): 2.5 | Top Diameter (IN): 3.9 | Product Dimension Type: Tapered & Graduated Product Dimensions | Product Dimensions: 3.9X4.7X2.5 IN | Pattern & Design: VB CUP COLD 16 OZ PET CLR 98 MM SERIES 20/50.'

#### Taxonomy prompts

In [72]:
model = 'gpt-4.1'
chunk_size = 32       
agent = OpenAIAgent(model=model, chunk_size=chunk_size)

In [ ]:
im_final.head(100)

In [73]:
im_final = cf.attribute_with_ai_optimized(im_final, agent, columns_for_description, prompt_options_string, example_desc, example_output, default_pack_size=1000)

Found 6 duplicate descriptions. Processing 1648 unique descriptions.


Running cost $2.8607: 100%|██████████| 52/52 [02:21<00:00,  2.72s/chunk]


extract_case_pack_with_examples: Processing 1648 unique descriptions instead of 1654 total descriptions


Running cost $3.7251: 100%|██████████| 52/52 [02:38<00:00,  3.04s/chunk]


#### Write file with just id and taxonomy

In [74]:
im_final.head()

,Entity--Item,Item Desc 1,Item Desc 2,Qty,Gross Cost,Net Cost,po_cost_amt,VB Flag,VGN,VPN,...,Compatible Product & Product Type,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Product Dimensions,Pattern & Design,Combined Descriptions,Description with Attributes,attributes,Case Pack
0,1--VBCLLH16DB,VB LID HOT CUP DOME BLK 92MM,PS 10 OZ SQUAT-24 OZ 20/50,10474,185100,185880,20.629195,Y - VB,Graphic Packaging,316873008,...,NaN,NaN,NaN,Standard Product Dimensions,NaN,NaN,VB LID HOT CUP DOME BLK 92MM PS 10 OZ SQUAT-24...,"Beverage Cup Type: Lid, Product Type Collapse...",Beverage Cup Type: Lid | Product Type Collapse...,1000
1,1--PTC09D92,VB CUP COLD 9 OZ SQUAT PET,92 SERIES CLR 20/50,15365,487038,339648,43.747324,Y - VB,"Carryout Bags, Inc.",VB-VG9OF,...,NaN,2.4,3.6,Tapered & Graduated Product Dimensions,3.6X2.8X2.4 IN,NaN,VB CUP COLD 9 OZ SQUAT PET 92 SERIES CLR 20/50,"Beverage Cup Type: Cup, Product Type Collapse...",Beverage Cup Type: Cup | Product Type Collapse...,1000
2,1--CPLUGBLACK,PLUG CIRCLE BLK FOR HOT CUP,LID SIP HOLE 5/400 BLACK,1507,37845,37845,24.190000,N,Amercareroyal,CPLUG-BK,...,NaN,NaN,NaN,NaN,NaN,NaN,PLUG CIRCLE BLK FOR HOT CUP LID SIP HOLE 5/400...,"Beverage Cup Type: Beverage Plug, Product Typ...",Beverage Cup Type: Beverage Plug | Product Typ...,2000
3,1--2CUPCARRY,CARRIER HOLDER FOR 2 CUP,MOLDED FIBER 8-24 OZ,247,8645,8645,36.050000,Y - Other,"Carryout Bags, Inc.",2CUPCARRY,...,NaN,NaN,NaN,NaN,NaN,NaN,CARRIER HOLDER FOR 2 CUP MOLDED FIBER 8-24 OZ,CARRIER HOLDER FOR 2 CUP MOLDED FIBER 8-24 OZ,Beverage Cup Type: | Product Type Collapse: C...,6000
4,1--DLKC12/20NH,LID CUP COLD DOME CLR PET,W/ NO HOLE KAL-CLR,412,27053,15549,70.922743,N,Pactiv,000000000009508059,...,Cup,NaN,NaN,Standard Product Dimensions,3.8X1.6 IN,NaN,LID CUP COLD DOME CLR PET W/ NO HOLE KAL-CLR,"Beverage Cup Type: Lid, Product Type Collapse...",Beverage Cup Type: Lid | Product Type Collapse...,1000


In [75]:
columns_for_description

['Beverage Cup Type',
 'Product Type Collapse',
 'Pack Size',
 'Color',
 'Material',
 'Foodservice Global Attributes',
 'Usage Temperature',
 'Beverage Cup Style',
 'Sustainable Products',
 'Product Capacity',
 'Compatible Product & Product Type',
 'Bottom Diameter (IN)',
 'Top Diameter (IN)',
 'Product Dimension Type',
 'Product Dimensions',
 'Pattern & Design']

In [76]:
im_final.tail(30)

,Entity--Item,Item Desc 1,Item Desc 2,Qty,Gross Cost,Net Cost,po_cost_amt,VB Flag,VGN,VPN,...,Compatible Product & Product Type,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Product Dimensions,Pattern & Design,Combined Descriptions,Description with Attributes,attributes,Case Pack
1624,1--MT696,6 OZ CLR 2 PC MARTINI,GLASS 96/CS,1,71,71,81.25,N,WNA,MT696,...,NaN,NaN,NaN,NaN,NaN,NaN,6 OZ CLR 2 PC MARTINI GLASS 96/CS,"Beverage Cup Type: Cup, Product Type Collapse...",Beverage Cup Type: Cup | Product Type Collapse...,96
1625,1--HC6NKPREM,CUP HOT PAPER KFT 6 OZ,BAMBOO FIBER 1000/CS,8,115,115,24.08,N,Lanca Sales Inc,HC6NKPREM,...,NaN,NaN,NaN,NaN,NaN,NaN,CUP HOT PAPER KFT 6 OZ BAMBOO FIBER 1000/CS,CUP HOT PAPER KFT 6 OZ BAMBOO FIBER 1000/CS,Beverage Cup Type: Cup | Product Type Collapse...,1000
1626,1--945LID1000,BIO LID TAPA 98.5,PET DOME HOLE 20/50CS,5,118,118,23.75,N,Carvajal Emaques Sa De Cv,945LID1000,...,NaN,NaN,NaN,NaN,NaN,NaN,BIO LID TAPA 98.5 PET DOME HOLE 20/50CS,BIO LID TAPA 98.5 PET DOME HOLE 20/50CS,Beverage Cup Type: Lid | Product Type Collapse...,1000
1627,1--FK-LRK1214,LID CUP COLD,FOR RK12/RK14,1,26,26,27.43,N,Pactiv,000000000009508216,...,Cup,NaN,NaN,Standard Product Dimensions,3.6X0.3 IN,NaN,LID CUP COLD FOR RK12/RK14,"Beverage Cup Type: Lid, Product Type Collapse...",Beverage Cup Type: Lid | Product Type Collapse...,1000
1628,1--SRHV7YAUCON,7OZ YAUCONO VENDING HOT CUP,NaN,70,3596,3596,51.42,N,Lanca Sales Inc,SR-HV7YAUCON,...,NaN,NaN,NaN,NaN,NaN,NaN,7OZ YAUCONO VENDING HOT CUP,7OZ YAUCONO VENDING HOT CUP,Beverage Cup Type: Cup | Product Type Collapse...,20000
1629,1--HARD816CUP,HARD EIGHT 16B16 OZ PRINTED,500/CS,140,3574,3574,25.55,N,Convermex Usa,2864224678,...,NaN,NaN,NaN,NaN,NaN,NaN,HARD EIGHT 16B16 OZ PRINTED 500/CS,HARD EIGHT 16B16 OZ PRINTED 500/CS,Beverage Cup Type: Cup | Product Type Collapse...,500
1630,1--KEBDL516N,LID DOME KARAT EARTH® BAGASSE,SIPPER FOR 10-24 OZ HOT CUP,32,640,640,20.00,N,Lollicup/Karat,KE-BDL516N,...,NaN,NaN,NaN,NaN,NaN,NaN,LID DOME KARAT EARTH® BAGASSE SIPPER FOR 10-24...,LID DOME KARAT EARTH® BAGASSE SIPPER FOR 10-24...,Beverage Cup Type: Lid | Product Type Collapse...,5000
1631,1--16HT,VB LID CUP HOT FLAT WHT,PS 10-24 OZ 10/100 WHITE,2,22,22,16.00,Y - VB,Graphic Packaging,316873002,...,Cup,NaN,NaN,NaN,NaN,NaN,VB LID CUP HOT FLAT WHT PS 10-24 OZ 10/100 WHITE,"Beverage Cup Type: Lid, Product Type Collapse...",Beverage Cup Type: Lid | Product Type Collapse...,1000
1632,1--322603023,LID STRAW SLOT 16-22 OZ LCRS22,IP TRANSLUCENT FLAT,1,36,36,30.05,N,Graphic Packaging,322603023,...,NaN,NaN,NaN,NaN,NaN,NaN,LID STRAW SLOT 16-22 OZ LCRS22 IP TRANSLUCENT ...,"Pack Size: 2000/Case, LID STRAW SLOT 16-22 OZ ...",Beverage Cup Type: Lid | Product Type Collapse...,2000
1633,1--210GCDW8K,CUP KFT PAPER 8 OZ DBL WALL,3.15 IN DIA 3.5 IN HT ECO,4,301,301,75.44,N,Pack N Wood,210GCDW8K,...,NaN,NaN,NaN,NaN,NaN,NaN,CUP KFT PAPER 8 OZ DBL WALL 3.15 IN DIA 3.5 IN...,"Beverage Cup Type: Cup, Product Type Collapse...",Beverage Cup Type: Cup | Product Type Collapse...,500


In [77]:
# Check if value is a BadRequestError or contains "BadRequestError" string
def has_error(value):
    # Check if it's an error object
    if hasattr(value, '__class__') and 'Error' in value.__class__.__name__:
        return True
    # Check if it's a string containing "BadRequestError" or "Error"
    if isinstance(value, str) and ('BadRequestError' in value or 'API Error' in value):
        return True
    return False

# Filter for rows with errors in either column
error_mask = im_final['attributes'].apply(has_error) | im_final['Case Pack'].apply(has_error)
error_rows = im_final[error_mask]

im_final[error_mask]

,Entity--Item,Item Desc 1,Item Desc 2,Qty,Gross Cost,Net Cost,po_cost_amt,VB Flag,VGN,VPN,...,Compatible Product & Product Type,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Product Dimensions,Pattern & Design,Combined Descriptions,Description with Attributes,attributes,Case Pack


In [78]:
im_final.to_csv('Stopped here.csv', index=False)

In [79]:
im_final_attributed, extract_attributes_to_dataframe = cf.extract_attributes_to_dataframe(im_final, columns_for_description, output_excel_filepath=f"{OP_PATH}{CATEGORY}_taxonomy.xlsx", vendor_col = 'VGN')

Checking for errors in attributes column...
Processing 1654 rows from the input DataFrame...
Adding Case Pack to attributes...
Successfully created DataFrame with 1654 rows and 19 columns.
Successfully wrote DataFrame to Excel: C:\Users\MGadupudi\PycharmProjects\ImperialDadeCategoryManagement\Data\Cups\Output\Cups_taxonomy.xlsx


In [80]:
cf.coverage_improvement(sfy, im_final, extract_attributes_to_dataframe, columns_for_description)

Computing coverage improvement analysis...
Computed post-LLM coverage for 19 columns
Computed initial coverage for 17 columns
Coverage improvement analysis complete. Found 16 columns to compare.


,% Coverage Post LLM,% Initial Coverage,Difference
Column_Name,,,
Beverage Cup Type,90.02%,57.13%,32.89%
Product Type Collapse,99.09%,64.51%,34.58%
Color,71.10%,42.74%,28.36%
Material,76.48%,48.25%,28.23%
Foodservice Global Attributes,49.70%,36.40%,13.30%
Usage Temperature,48.97%,30.11%,18.86%
Beverage Cup Style,22.31%,11.85%,10.46%
Sustainable Products,15.11%,11.25%,3.87%
Product Capacity,82.89%,42.99%,39.90%


#### write file with attributes

In [81]:
# Filter out rows where Entity is '1' (keeping all other entities)
im_final = im_final[im_final['Entity--Item'].str.startswith('1--')]

print(f"Rows remaining after filtering out Entity 1: {len(im_final)}")

Rows remaining after filtering out Entity 1: 1654


In [82]:
im_final_attributed.to_csv(f"{OP_PATH}{CATEGORY}_Attributed.csv", index=False)

In [ ]:
im_final_attributed = pd.read_csv(f'{OP_PATH}{CATEGORY}_Attributed.csv')

In [ ]:
im_final_attributed[im_final_attributed['Entity--Item'] == '1--KON8DW']

In [ ]:
item_df_real_time[item_df_real_time['Printed Item flag'] == 'Y'].head()

In [85]:
item_df_real_time['concat_key'] = (
    item_df_real_time['Company No.'].astype(str).str.strip() +
    '--' +
    item_df_real_time['Item Code'].astype(str).str.strip()
)

In [86]:
item_df_real_time[item_df_real_time['concat_key'] == '1--KON8DW']

,Company No.,Location,Region,Warehouse Code,Warehouse Name,Item Code,Item Description 1,Item Description 2,Item Description 3,Purchasing UoM,...,Daylight_Savings_Flag,Entity_Name,Warehouse_Name,Entity_Name_Old,Entity_Name_New,Whs_Name_Old,Whs_Name_New,Item Sub Category,Warehouse No.,concat_key
1158426,1,Jersey City,Northeast,JC,Jersey City,KON8DW,KONDITONI 8 OZ DOUBLE WALL,PTD,KON8DW KON8DW,CS,...,Y,Imperial Dade,Jersey City,Imperial Dade,Imperial Dade,Jersey City,Jersey City,Printed Merch,1--JC,1--KON8DW
1975755,1,Bordentown,Northeast,NJBT,Bordentown,KON8DW,KONDITONI 8 OZ DOUBLE WALL,PTD,KON8DW KON8DW,CS,...,Y,Imperial Dade,Bordentown,Imperial Dade,Imperial Dade,Bordentown,Bordentown,Printed Merch,1--NJBT,1--KON8DW


In [89]:
item_test = item_df_real_time[item_df_real_time['Printed Item flag'] == 'Y']

In [90]:
item_test.shape

(283276, 78)

In [91]:
# Strip the Entity--Item column in im_final_attributed for consistent comparison
im_final_attributed['Entity--Item'] = im_final_attributed['Entity--Item'].astype(str).str.strip()

In [92]:
# Filter im_final_attributed to keep only rows where Entity--Item exists in the concat_key
im_final_attributed_filtered = im_final_attributed[
    ~im_final_attributed['Entity--Item'].isin(item_test['concat_key'])
]

print(f"Original rows: {len(im_final_attributed)}")
print(f"Filtered rows: {len(im_final_attributed_filtered)}")

Original rows: 1654
Filtered rows: 1653


In [96]:
im_final_attributed_filtered[im_final_attributed_filtered['Entity--Item'] == '1--45unipack']

,Entity--Item,Item Desc 1,Item Desc 2,Qty,Gross Cost,Net Cost,po_cost_amt,VB Flag,VGN,VPN,...,Beverage Cup Style,Sustainable Products,Product Capacity,Compatible Product & Product Type,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Product Dimensions,Pattern & Design,Case Pack


In [93]:
im_final_attributed_filtered.to_csv(f"{OP_PATH}{CATEGORY}_Attributed_non_printed.csv", index=False)

In [ ]:
item_df_real_time.head()

In [ ]:
item_df_real_time[item_df_real_time['concat_key'] == '1--KON8DW']

In [94]:
im_final_attributed_filtered.head()

,Entity--Item,Item Desc 1,Item Desc 2,Qty,Gross Cost,Net Cost,po_cost_amt,VB Flag,VGN,VPN,...,Beverage Cup Style,Sustainable Products,Product Capacity,Compatible Product & Product Type,Bottom Diameter (IN),Top Diameter (IN),Product Dimension Type,Product Dimensions,Pattern & Design,Case Pack
0,1--VBCLLH16DB,VB LID HOT CUP DOME BLK 92MM,PS 10 OZ SQUAT-24 OZ 20/50,10474,185100,185880,20.629195,Y - VB,Graphic Packaging,316873008,...,,Yes,16 OZ,10 OZ Squat-24 OZ Cups,,92MM,Standard Product Dimensions,,VB LID HOT CUP DOME BLK 92MM PS 10 OZ SQUAT-24...,1000
1,1--PTC09D92,VB CUP COLD 9 OZ SQUAT PET,92 SERIES CLR 20/50,15365,487038,339648,43.747324,Y - VB,"Carryout Bags, Inc.",VB-VG9OF,...,,,9 OZ,,2.4,3.6,Tapered & Graduated Product Dimensions,3.6X2.8X2.4 IN,VB CUP COLD 9 OZ SQUAT PET 92 SERIES CLR 20/50,1000
2,1--CPLUGBLACK,PLUG CIRCLE BLK FOR HOT CUP,LID SIP HOLE 5/400 BLACK,1507,37845,37845,24.190000,N,Amercareroyal,CPLUG-BK,...,,,,Hot Cup Lid,,,,,PLUG CIRCLE BLK FOR HOT CUP LID SIP HOLE 5/400...,2000
3,1--2CUPCARRY,CARRIER HOLDER FOR 2 CUP,MOLDED FIBER 8-24 OZ,247,8645,8645,36.050000,Y - Other,"Carryout Bags, Inc.",2CUPCARRY,...,,,8-24 OZ,Cup,,,,,,6000
4,1--DLKC12/20NH,LID CUP COLD DOME CLR PET,W/ NO HOLE KAL-CLR,412,27053,15549,70.922743,N,Pactiv,000000000009508059,...,,,,Cup,,,Standard Product Dimensions,3.8X1.6 IN,LID CUP COLD DOME CLR PET W/ NO HOLE KAL-CLR,1000
